# E-Commerce Order Management Portal

## Introduction
The E-Commerce Order Management Portal is a Python-based
system designed to manage products, inventory and customer
orders for an online shopping business.

## Main Features
- Admin Login
- Product Management
- Inventory Management
- Customer Order Placement
- Order Searching
- Order Status Management
- Order Cancellation
- Sales Reports
- Data Backup

## Technologies Used
- Python
- Jupyter Notebook
- JSON File Handling

In [1]:
import json
import os
import shutil
from datetime import datetime

# Database file
DATA_FILE = "ecommerce_data.json"

# Backup folder
BACKUP_FOLDER = "backup"

# Admin credentials
ADMIN_USERNAME = "admin"
ADMIN_PASSWORD = "admin123"

# Order statuses
ORDER_STATUSES = [
    "Pending",
    "Processing",
    "Shipped",
    "Delivered"
]

print("E-Commerce Order Management Portal")
print("System initialized successfully.")

E-Commerce Order Management Portal
System initialized successfully.


In [2]:
{
    "products": [],
    "orders": []
}

{'products': [], 'orders': []}

In [3]:
def load_data():

    try:
        with open(DATA_FILE, "r") as file:
            data = json.load(file)

            if not isinstance(data, dict):
                raise ValueError("Invalid database format.")

            if not isinstance(data.get("products"), list):
                raise ValueError("Invalid product records.")

            if not isinstance(data.get("orders"), list):
                raise ValueError("Invalid order records.")

            return data

    except FileNotFoundError:

        print("Database not found. Creating a new database.")

        data = {
            "products": [],
            "orders": []
        }

        save_data(data)
        return data

    except json.JSONDecodeError:

        print("Error: Database contains invalid JSON.")
        return {"products": [], "orders": []}

    except (PermissionError, OSError, ValueError) as error:

        print("Database error:", error)
        return {"products": [], "orders": []}


def save_data(data):

    try:
        with open(DATA_FILE, "w") as file:
            json.dump(data, file, indent=4)

        return True

    except (PermissionError, OSError) as error:

        print("Unable to save database:", error)
        return False


# Load existing database
data = load_data()

print("Total Products:", len(data["products"]))
print("Total Orders:", len(data["orders"]))

Database not found. Creating a new database.
Total Products: 0
Total Orders: 0


In [4]:
def user_login(username, password):

    if username == ADMIN_USERNAME and password == ADMIN_PASSWORD:

        print("\nLogin successful!")
        print("Welcome to E-Commerce Order Portal.")

        return True

    else:

        print("\nInvalid username or password.")

        return False


# Test login
user_login("admin", "admin123")


Login successful!
Welcome to E-Commerce Order Portal.


True

In [5]:
def add_product(data):

    print("\n--- ADD NEW PRODUCT ---")

    product_id = input("Enter Product ID: ").strip()

    if product_id == "":
        print("Product ID cannot be empty.")
        return

    # Check duplicate product ID
    for product in data["products"]:

        if product["product_id"].lower() == product_id.lower():

            print("Product ID already exists.")
            return

    name = input("Enter Product Name: ").strip()
    category = input("Enter Category: ").strip()

    if name == "" or category == "":
        print("Name and category cannot be empty.")
        return

    try:

        price = float(input("Enter Product Price: "))
        stock = int(input("Enter Available Stock: "))

        if price <= 0:
            print("Price must be greater than zero.")
            return

        if stock < 0:
            print("Stock cannot be negative.")
            return

        new_product = {
            "product_id": product_id,
            "name": name,
            "category": category,
            "price": price,
            "stock": stock
        }

        data["products"].append(new_product)

        if save_data(data):

            print("Product added successfully!")

        else:

            data["products"].pop()
            print("Product could not be saved.")

    except ValueError:

        print("Invalid input! Price and stock must be numeric.")

In [6]:
def view_products(data):

    print("\n--- AVAILABLE PRODUCTS ---")

    if len(data["products"]) == 0:

        print("No products available.")
        return

    for product in data["products"]:

        print("--------------------------")

        print("Product ID:", product["product_id"])
        print("Name:", product["name"])
        print("Category:", product["category"])
        print("Price: Rs.", product["price"])
        print("Available Stock:", product["stock"])

    print("--------------------------")
    print("Total Products:", len(data["products"]))

In [7]:
def search_product(data, product_id):

    for product in data["products"]:

        if product["product_id"].lower() == product_id.lower():

            print("\nProduct Found!")

            print("Product ID:", product["product_id"])
            print("Name:", product["name"])
            print("Category:", product["category"])
            print("Price: Rs.", product["price"])
            print("Stock:", product["stock"])

            return product

    print("Product not found.")

    return None

In [8]:
def update_inventory(data, product_id):

    for product in data["products"]:

        if product["product_id"].lower() == product_id.lower():

            print("\n1. Update Price")
            print("2. Update Stock")

            try:

                choice = int(input("Select Option: "))

                if choice == 1:

                    new_price = float(input("Enter New Price: "))

                    if new_price <= 0:
                        print("Price must be positive.")
                        return

                    old_value = product["price"]
                    product["price"] = new_price

                elif choice == 2:

                    new_stock = int(input("Enter New Stock: "))

                    if new_stock < 0:
                        print("Stock cannot be negative.")
                        return

                    old_value = product["stock"]
                    product["stock"] = new_stock

                else:

                    print("Invalid choice.")
                    return

                if save_data(data):

                    print("Product information updated successfully!")

                else:

                    if choice == 1:
                        product["price"] = old_value
                    else:
                        product["stock"] = old_value

            except ValueError:

                print("Please enter a valid number.")

            return

    print("Product not found.")

In [9]:
def place_order(data):

    print("\n--- PLACE CUSTOMER ORDER ---")

    customer_name = input("Enter Customer Name: ").strip()

    if customer_name == "":
        print("Customer name cannot be empty.")
        return

    cart = []

    view_products(data)

    if len(data["products"]) == 0:
        return

    while True:

        product_id = input(
            "\nEnter Product ID (or type done): "
        ).strip()

        if product_id.lower() == "done":
            break

        selected_product = None

        for product in data["products"]:

            if product["product_id"].lower() == product_id.lower():

                selected_product = product
                break

        if selected_product is None:

            print("Product not found.")
            continue

        try:

            quantity = int(input("Enter Quantity: "))

            if quantity <= 0:
                print("Quantity must be greater than zero.")
                continue

            already_selected = 0

            for item in cart:

                if item["product_id"] == selected_product["product_id"]:
                    already_selected += item["quantity"]

            if quantity + already_selected > selected_product["stock"]:

                print("Insufficient stock available.")
                continue

            item = {
                "product_id": selected_product["product_id"],
                "name": selected_product["name"],
                "price": selected_product["price"],
                "quantity": quantity,
                "subtotal": selected_product["price"] * quantity
            }

            cart.append(item)

            print("Product added to order.")

        except ValueError:

            print("Please enter a valid quantity.")

    if len(cart) == 0:

        print("No products selected. Order cancelled.")
        return

    total = 0

    for item in cart:
        total += item["subtotal"]

    print("\nOrder Summary")

    for item in cart:

        print(
            item["name"],
            "x", item["quantity"],
            "= Rs.", item["subtotal"]
        )

    print("Total Amount: Rs.", round(total, 2))

    confirm = input("Confirm Order? (yes/no): ").lower()

    if confirm != "yes":

        print("Order cancelled.")
        return

    order_number = len(data["orders"]) + 1
    order_id = "ORD" + str(order_number).zfill(4)

    order = {
        "order_id": order_id,
        "customer_name": customer_name,
        "items": cart,
        "total": total,
        "status": "Pending",
        "date": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }

    # Reduce inventory
    for item in cart:

        for product in data["products"]:

            if product["product_id"] == item["product_id"]:

                product["stock"] -= item["quantity"]

    data["orders"].append(order)

    if save_data(data):

        print("\nOrder placed successfully!")
        print("Order ID:", order_id)
        print("Total Payment: Rs.", round(total, 2))

    else:

        data["orders"].pop()

        # Restore inventory if saving fails
        for item in cart:

            for product in data["products"]:

                if product["product_id"] == item["product_id"]:

                    product["stock"] += item["quantity"]

        print("Order could not be saved.")

In [10]:
def search_order(data, order_id):

    for order in data["orders"]:

        if order["order_id"].lower() == order_id.lower():

            print("\n--- ORDER DETAILS ---")

            print("Order ID:", order["order_id"])
            print("Customer:", order["customer_name"])
            print("Date:", order["date"])
            print("Status:", order["status"])

            print("\nOrdered Products:")

            for item in order["items"]:

                print(
                    item["name"],
                    "Quantity:", item["quantity"],
                    "Subtotal: Rs.", item["subtotal"]
                )

            print("\nTotal: Rs.", order["total"])

            return order

    print("Order not found.")

    return None

In [11]:
def update_order_status(data, order_id):

    for order in data["orders"]:

        if order["order_id"].lower() == order_id.lower():

            print("\nCurrent Status:", order["status"])

            print("\nAvailable Statuses:")

            for i, status in enumerate(ORDER_STATUSES, start=1):

                print(i, ".", status)

            try:

                choice = int(input("Select New Status: "))

                if choice < 1 or choice > len(ORDER_STATUSES):

                    print("Invalid status selection.")
                    return

                if order["status"] == "Cancelled":

                    print("Cancelled orders cannot be updated.")
                    return

                if order["status"] == "Delivered":

                    print("Delivered orders cannot be updated.")
                    return

                old_status = order["status"]

                order["status"] = ORDER_STATUSES[choice - 1]

                if save_data(data):

                    print("Order status updated successfully!")

                else:

                    order["status"] = old_status

            except ValueError:

                print("Please enter a valid number.")

            return

    print("Order not found.")

In [12]:
def cancel_order(data, order_id):

    for order in data["orders"]:

        if order["order_id"].lower() == order_id.lower():

            if order["status"] in ["Cancelled", "Delivered"]:

                print("This order cannot be cancelled.")
                return

            confirm = input(
                "Are you sure you want to cancel? (yes/no): "
            ).lower()

            if confirm == "yes":

                old_status = order["status"]

                order["status"] = "Cancelled"

                # Restore product stock
                for item in order["items"]:

                    for product in data["products"]:

                        if product["product_id"] == item["product_id"]:

                            product["stock"] += item["quantity"]

                if save_data(data):

                    print("Order cancelled successfully.")
                    print("Inventory has been restored.")

                else:

                    order["status"] = old_status

                    for item in order["items"]:

                        for product in data["products"]:

                            if product["product_id"] == item["product_id"]:

                                product["stock"] -= item["quantity"]

                    print("Cancellation could not be saved.")

            else:

                print("Cancellation stopped.")

            return

    print("Order not found.")

In [13]:
def generate_sales_report(data):

    print("\n========== SALES REPORT ==========")

    total_orders = len(data["orders"])

    completed_orders = 0
    pending_orders = 0
    cancelled_orders = 0
    total_revenue = 0

    for order in data["orders"]:

        if order["status"] == "Delivered":

            completed_orders += 1
            total_revenue += order["total"]

        elif order["status"] == "Cancelled":

            cancelled_orders += 1

        else:

            pending_orders += 1

    print("Total Orders:", total_orders)
    print("Delivered Orders:", completed_orders)
    print("Pending/Processing/Shipped:", pending_orders)
    print("Cancelled Orders:", cancelled_orders)

    print(
        "Delivered Sales Revenue: Rs.",
        round(total_revenue, 2)
    )

    print("=================================")

    return total_revenue

In [14]:
def inventory_report(data):

    print("\n========== INVENTORY REPORT ==========")

    total_products = len(data["products"])
    total_units = 0
    low_stock = 0

    for product in data["products"]:

        total_units += product["stock"]

        if product["stock"] <= 5:

            low_stock += 1

            print(
                "Low Stock:",
                product["name"],
                "- Remaining:", product["stock"]
            )

    print("\nTotal Products:", total_products)
    print("Total Units Available:", total_units)
    print("Products with Low Stock:", low_stock)

    print("======================================")

In [15]:
def backup_data():

    try:

        if not os.path.exists(DATA_FILE):

            print("Database file does not exist.")
            return False

        os.makedirs(BACKUP_FOLDER, exist_ok=True)

        backup_file = os.path.join(
            BACKUP_FOLDER,
            "ecommerce_backup.json"
        )

        shutil.copy2(DATA_FILE, backup_file)

        print("Backup created successfully!")
        print("Location:", backup_file)

        return True

    except (FileNotFoundError, PermissionError, OSError) as error:

        print("Backup error:", error)
        return False

In [16]:
def main():

    data = load_data()

    print("\n===================================")
    print("     E-COMMERCE ORDER PORTAL")
    print("===================================")

    login_success = False

    # Three login attempts
    for attempt in range(3):

        username = input("\nEnter Username: ")
        password = input("Enter Password: ")

        if user_login(username, password):

            login_success = True
            break

        else:

            print("Attempts remaining:", 2 - attempt)

    if not login_success:

        print("Access denied.")
        return

    try:

        while True:

            print("\n========== MAIN MENU ==========")
            print("1. Add New Product")
            print("2. View All Products")
            print("3. Search Product")
            print("4. Update Inventory")
            print("5. Place Customer Order")
            print("6. Search Order")
            print("7. Update Order Status")
            print("8. Cancel Order")
            print("9. Generate Sales Report")
            print("10. Inventory Report")
            print("11. Create Data Backup")
            print("12. Exit")
            print("===============================")

            try:

                choice = int(input("Enter Your Choice: "))

                if choice == 1:

                    add_product(data)

                elif choice == 2:

                    view_products(data)

                elif choice == 3:

                    product_id = input("Enter Product ID: ")
                    search_product(data, product_id)

                elif choice == 4:

                    product_id = input("Enter Product ID: ")
                    update_inventory(data, product_id)

                elif choice == 5:

                    place_order(data)

                elif choice == 6:

                    order_id = input("Enter Order ID: ")
                    search_order(data, order_id)

                elif choice == 7:

                    order_id = input("Enter Order ID: ")
                    update_order_status(data, order_id)

                elif choice == 8:

                    order_id = input("Enter Order ID: ")
                    cancel_order(data, order_id)

                elif choice == 9:

                    generate_sales_report(data)

                elif choice == 10:

                    inventory_report(data)

                elif choice == 11:

                    backup_data()

                elif choice == 12:

                    print("Thank you for using E-Commerce Order Portal.")
                    break

                else:

                    print("Invalid choice. Select between 1 and 12.")

            except ValueError:

                print("Invalid input. Please enter a number.")

    except (KeyboardInterrupt, EOFError):

        print("\nPortal interrupted.")

    finally:

        print("\nPortal session closed.")


print("Main function created successfully.")

Main function created successfully.


In [17]:
main()


     E-COMMERCE ORDER PORTAL



Enter Username:  admin
Enter Password:  admin123



Login successful!
Welcome to E-Commerce Order Portal.

========== MAIN MENU ==========
1. Add New Product
2. View All Products
3. Search Product
4. Update Inventory
5. Place Customer Order
6. Search Order
7. Update Order Status
8. Cancel Order
9. Generate Sales Report
10. Inventory Report
11. Create Data Backup
12. Exit


Enter Your Choice:  1



--- ADD NEW PRODUCT ---


Enter Product ID:  PO 1
Enter Product Name:  Mouse
Enter Category:  Electronics
Enter Product Price:  2500
Enter Available Stock:  12


... (69 outputs left out)

Enter Your Choice:  11


Backup created successfully!
Location: backup\ecommerce_backup.json

========== MAIN MENU ==========
1. Add New Product
2. View All Products
3. Search Product
4. Update Inventory
5. Place Customer Order
6. Search Order
7. Update Order Status
8. Cancel Order
9. Generate Sales Report
10. Inventory Report
11. Create Data Backup
12. Exit


Enter Your Choice:  12


Thank you for using E-Commerce Order Portal.

Portal session closed.
